In [1]:
%pip install phonenumbers --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import phonenumbers

Xử lý cho czechia

In [3]:
import pandas as pd

# Đường dẫn tới 2 file
gd_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv"
checkpoint_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\raw_country_extracted\Germany.csv"

# 1. Đọc dữ liệu
df_checkpoint = pd.read_csv(checkpoint_path)
df_gd = pd.read_csv(gd_path)

In [4]:
df_checkpoint.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7525 entries, 0 to 7524
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   post_title          7525 non-null   object 
 1   address             7525 non-null   object 
 2   latitude            7525 non-null   float64
 3   longitude           7525 non-null   float64
 4   phone               7333 non-null   object 
 5   website             5508 non-null   object 
 6   facebook            0 non-null      float64
 7   instagram           0 non-null      float64
 8   twitter             0 non-null      float64
 9   post_content        7525 non-null   object 
 10  google_maps_link    7525 non-null   object 
 11  google_profile      7525 non-null   object 
 12  google_review_link  7525 non-null   object 
 13  country             7525 non-null   object 
dtypes: float64(5), object(9)
memory usage: 823.2+ KB


In [5]:
# --- Kiểm tra định dạng số điện thoại trước khi chuẩn hóa ---
phones = df_checkpoint["phone"].astype(str).str.strip()

# Loại bỏ các nan
valid_phones = phones[~phones.str.lower().isin(["nan", "none", ""]) & (phones != "")]

# Kiểm tra định dạng số điện thoại
# Có bắt đầu bằng '00'?
count_00 = valid_phones.str.startswith("00").sum()
# Có bắt đầu bằng '+' ?
count_plus = valid_phones.str.startswith("+").sum()
# Có dấu '-'?
count_dash = valid_phones.str.contains("-", regex=False).sum()
# Có dấu cách ?
count_space = valid_phones.str.contains(" ", regex=False).sum()

# Kiểm tra range số điện thoại
phones_cleaned = valid_phones.str.replace(r"[-\s]", "", regex=True)
lengths = phones_cleaned.str.len()
min_len = lengths.min()
max_len = lengths.max()

# --- In kết quả ---
print(f"Số bắt đầu bằng '00': {count_00}")
print(f"Số bắt đầu bằng '+': {count_plus}")
print(f"Số chứa dấu '-': {count_dash}")
print(f"Số chứa dấu cách: {count_space}")
print(f"Độ dài ngắn nhất (sau khi loại bỏ ký tự và NaN): {min_len}")
print(f"Độ dài dài nhất: {max_len}")


Số bắt đầu bằng '00': 0
Số bắt đầu bằng '+': 0
Số chứa dấu '-': 1
Số chứa dấu cách: 7333
Độ dài ngắn nhất (sau khi loại bỏ ký tự và NaN): 8
Độ dài dài nhất: 13


In [6]:
# Chuẩn hóa số điện thoại theo tiêu chuẩn E.164
from phonenumbers import PhoneNumberFormat
# ---- Country to Region Code Mapping ----
country_region_map = {
    "Sweden": "SE",
    "United Kingdom": "GB",
    "France": "FR",
    "Germany": "DE",
    "Poland": "PL",
    "Czechia": "CZ",
    "Slovakia": "SK",
    "Italy": "IT",
    "Spain": "ES",
    "Portugal": "PT",
    "Belgium": "BE",
    "Netherlands": "NL",
    "Hungary": "HU",
    "Austria": "AT"
}

# Hàm clean - giữ nan và xóa kí tự lạ (gồm dấu cách và - )
def clean_phone_number(raw_phone):
    if pd.isna(raw_phone):
        return raw_phone  
    raw_phone = str(raw_phone)
    cleaned = re.sub(r'[^\d+]', '', raw_phone)  
    return cleaned

# Chuẩn hóa số hợp lệ theo E.164 (thư viện phonenumbers để đưa về dạng sđt quốc tế)
def standardize_phone_number(row):
    raw = clean_phone_number(row["phone"])
    region = country_region_map.get(row["country"], None)
    if pd.isna(raw) or not str(raw).strip():
        return row["phone"]  
    try:
        parsed = phonenumbers.parse(raw, region)
        if phonenumbers.is_valid_number(parsed):
            return phonenumbers.format_number(parsed, PhoneNumberFormat.E164)
        else:
            return row["phone"]
    except:
        return row["phone"]

# Gán nhãn hợp lệ / không hợp lệ / thiếu để tiện lọc thủ công (nếu có)
def label_phone_status(row):
    raw = clean_phone_number(row["phone"])
    region = country_region_map.get(row["country"], None)
    if pd.isna(raw) or not str(raw).strip():
        return pd.NA 
    try:
        parsed = phonenumbers.parse(raw, region)
        if phonenumbers.is_valid_number(parsed):
            return 1  # Valid
        else:
            return 0  # Invalid
    except:
        return 0  # Invalid do lỗi

# Áp dụng hàm
df_checkpoint["phone"] = df_checkpoint.apply(standardize_phone_number, axis=1)
df_checkpoint["phone_status"] = df_checkpoint.apply(label_phone_status, axis=1)
print(df_checkpoint[["phone", "country", "phone_status"]].head(5))

            phone  country phone_status
0  +4935816862292  Germany            1
1  +4935818767676  Germany            1
2   +493581767971  Germany            1
3  +4935813228112  Germany            1
4   +493584229927  Germany            1


In [7]:
df_checkpoint.head(5)

,post_title,address,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status
0,Vietnam Quan,"Lausitzer Str. 20, 02828 Görlitz, Germany",51.170768,14.978383,+4935816862292,NaN,NaN,NaN,NaN,"Vietnam Quan located in Lausitzer Str. 20, 028...",https://maps.google.com/?cid=7777768487273170008,https://maps.google.com/?q=place_id:ChIJ-dnDxg...,https://search.google.com/local/reviews?placei...,Germany,1
1,Xin Chào Restaurant,"Hildegard-Burjan-Platz 1, 02826 Görlitz, Germany",51.156261,14.982172,+4935818767676,https://www.lieferando.de/speisekarte/xin-chao...,NaN,NaN,NaN,Xin Chào Restaurant located in Hildegard-Burja...,https://maps.google.com/?cid=15842596843979238522,https://maps.google.com/?q=place_id:ChIJ69bbW3...,https://search.google.com/local/reviews?placei...,Germany,1
2,Vietnam-Wok Bistro,"Berliner Str. 17, 02826 Görlitz, Germany",51.149755,14.982371,+493581767971,NaN,NaN,NaN,NaN,Vietnam-Wok Bistro located in Berliner Str. 17...,https://maps.google.com/?cid=13039952271823855627,https://maps.google.com/?q=place_id:ChIJAeQxdC...,https://search.google.com/local/reviews?placei...,Germany,1
3,MEKONG RESTAURANT,"Brautwiesenstraße 04, 02826 Görlitz, Germany",51.151013,14.969034,+4935813228112,http://www.mekongrestaurant.de/,NaN,NaN,NaN,We're under construction.Please check back for...,https://maps.google.com/?cid=10683515021689842950,https://maps.google.com/?q=place_id:ChIJ08s1gs...,https://search.google.com/local/reviews?placei...,Germany,1
4,Vietnam-Imbiss,"Bahnhofstraße 2, 02791 Oderwitz, Germany",50.952569,14.726453,+493584229927,NaN,NaN,NaN,NaN,"Vietnam-Imbiss located in Bahnhofstraße 2, 027...",https://maps.google.com/?cid=15742013323567481727,https://maps.google.com/?q=place_id:ChIJTfl0lK...,https://search.google.com/local/reviews?placei...,Germany,1


In [8]:
# tách address thành: street (có zip), zip (postcode), city
def robust_split_address(address, country="Germany"):
    if pd.isna(address):
        return "", "", ""

    # Bước 1: Xoá phần country ở cuối
    address = re.sub(
        rf'[,\s]*{re.escape(str(country))}[\s,\d]*$', '', address, flags=re.IGNORECASE
    ).strip()

    # Bước 2: Tìm postcode và city (giả định postcode là 123 45 hoặc 12345 hoặc 123-45)
    match = re.search(r'(\d{3}[-\s]?\d{2})\s+(.+)$', address)
    if match:
        zip_code = match.group(1).strip()
        city = match.group(2).strip()

        # Street là phần trước ZIP (có thể kèm dấu phẩy)
        street_part = address[:match.start()].strip().rstrip(', ')
        street_with_zip = f"{street_part}, {zip_code}"
        return street_with_zip, zip_code, city

    # Nếu không tách được, coi toàn bộ là street
    return address, "", ""

# Áp dụng vào dữ liệu ban đầu
df_checkpoint[['street', 'zip', 'city']] = df_checkpoint['address'].apply(
    lambda x: pd.Series(robust_split_address(x))
)
# xóa cột address
df_checkpoint.drop(columns=['address'], inplace=True)


In [9]:
df_checkpoint.head(5)

,post_title,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status,street,zip,city
0,Vietnam Quan,51.170768,14.978383,+4935816862292,NaN,NaN,NaN,NaN,"Vietnam Quan located in Lausitzer Str. 20, 028...",https://maps.google.com/?cid=7777768487273170008,https://maps.google.com/?q=place_id:ChIJ-dnDxg...,https://search.google.com/local/reviews?placei...,Germany,1,"Lausitzer Str. 20, 02828",02828,Görlitz
1,Xin Chào Restaurant,51.156261,14.982172,+4935818767676,https://www.lieferando.de/speisekarte/xin-chao...,NaN,NaN,NaN,Xin Chào Restaurant located in Hildegard-Burja...,https://maps.google.com/?cid=15842596843979238522,https://maps.google.com/?q=place_id:ChIJ69bbW3...,https://search.google.com/local/reviews?placei...,Germany,1,"Hildegard-Burjan-Platz 1, 02826",02826,Görlitz
2,Vietnam-Wok Bistro,51.149755,14.982371,+493581767971,NaN,NaN,NaN,NaN,Vietnam-Wok Bistro located in Berliner Str. 17...,https://maps.google.com/?cid=13039952271823855627,https://maps.google.com/?q=place_id:ChIJAeQxdC...,https://search.google.com/local/reviews?placei...,Germany,1,"Berliner Str. 17, 02826",02826,Görlitz
3,MEKONG RESTAURANT,51.151013,14.969034,+4935813228112,http://www.mekongrestaurant.de/,NaN,NaN,NaN,We're under construction.Please check back for...,https://maps.google.com/?cid=10683515021689842950,https://maps.google.com/?q=place_id:ChIJ08s1gs...,https://search.google.com/local/reviews?placei...,Germany,1,"Brautwiesenstraße 04, 02826",02826,Görlitz
4,Vietnam-Imbiss,50.952569,14.726453,+493584229927,NaN,NaN,NaN,NaN,"Vietnam-Imbiss located in Bahnhofstraße 2, 027...",https://maps.google.com/?cid=15742013323567481727,https://maps.google.com/?q=place_id:ChIJTfl0lK...,https://search.google.com/local/reviews?placei...,Germany,1,"Bahnhofstraße 2, 02791",02791,Oderwitz


In [10]:
# Kiểm tra active của web 
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor

# Chuẩn hóa URL
def normalize_url(url):
    if pd.isna(url) or not str(url).strip():
        return None
    url = url.strip()
    parsed = urlparse(url)
    if not parsed.scheme:
        return "http://" + url
    return url

# Kiểm tra hoạt động website, giữ original URL
def check_url(original_url):
    norm_url = normalize_url(original_url)
    if not norm_url:
        return (original_url, None, None, False)
    try:
        response = requests.get(norm_url, timeout=3, allow_redirects=True)
        final_url = response.url
        status = response.status_code
        is_active = 200 <= status < 400
        return (original_url, status, final_url, is_active)
    except:
        return (original_url, 0, None, False)

# Áp dụng đa luồng
df_checkpoint["normalized_url"] = df_checkpoint["website"].apply(normalize_url)
urls = df_checkpoint["normalized_url"].tolist()

with ThreadPoolExecutor(max_workers=30) as executor:
    results = list(executor.map(check_url, urls))

# Ghi kết quả vào DataFrame
df_checkpoint["web_status"] = [r[1] for r in results]
df_checkpoint["final_url"] = [r[2] for r in results]
df_checkpoint["is_active"] = [r[3] for r in results]


In [11]:
# Kiểm tra các link mạng xã hội bị lẫn trong website
# Xóa cột normalized_url
df_checkpoint.drop(columns=["normalized_url"], inplace=True)

# Xác định nền tảng mạng xã hội 
def classify_social_platform(url):
    if pd.isna(url):
        return None
    url = url.lower()
    if "facebook.com" in url:
        return "facebook"
    elif "instagram.com" in url:
        return "instagram"
    elif "twitter.com" in url or "x.com" in url:
        return "twitter"
    return None

df_checkpoint["social_platform"] = df_checkpoint["website"].apply(classify_social_platform)

# Chuyển các link sai về đúng cột
for platform in ["facebook", "instagram", "twitter"]:
    df_checkpoint[platform] = df_checkpoint.apply(
        lambda row: row["website"] if row["social_platform"] == platform and pd.isna(row[platform]) else row[platform],
        axis=1
    )

# Lưu kết quả vào file CSV
output_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\Clean 25-5 - raw\germany_web_check.csv"

In [12]:
# Xoá khỏi giá trị cột website nếu là link MXH 
df_checkpoint.loc[df_checkpoint["social_platform"].notna(), "website"] = None
df_checkpoint.drop(columns=["social_platform"], inplace=True)

In [13]:
df_checkpoint.head(3)


,post_title,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status,street,zip,city,web_status,final_url,is_active
0,Vietnam Quan,51.170768,14.978383,+4935816862292,NaN,NaN,NaN,NaN,"Vietnam Quan located in Lausitzer Str. 20, 028...",https://maps.google.com/?cid=7777768487273170008,https://maps.google.com/?q=place_id:ChIJ-dnDxg...,https://search.google.com/local/reviews?placei...,Germany,1,"Lausitzer Str. 20, 02828",02828,Görlitz,NaN,None,False
1,Xin Chào Restaurant,51.156261,14.982172,+4935818767676,https://www.lieferando.de/speisekarte/xin-chao...,NaN,NaN,NaN,Xin Chào Restaurant located in Hildegard-Burja...,https://maps.google.com/?cid=15842596843979238522,https://maps.google.com/?q=place_id:ChIJ69bbW3...,https://search.google.com/local/reviews?placei...,Germany,1,"Hildegard-Burjan-Platz 1, 02826",02826,Görlitz,403.0,https://www.lieferando.de/speisekarte/xin-chao...,False
2,Vietnam-Wok Bistro,51.149755,14.982371,+493581767971,NaN,NaN,NaN,NaN,Vietnam-Wok Bistro located in Berliner Str. 17...,https://maps.google.com/?cid=13039952271823855627,https://maps.google.com/?q=place_id:ChIJAeQxdC...,https://search.google.com/local/reviews?placei...,Germany,1,"Berliner Str. 17, 02826",02826,Görlitz,NaN,None,False


In [14]:
# tạo copy
df_checkpoint_copy = df_checkpoint.copy()

In [15]:
# 7. Lấy danh sách cột chuẩn từ file gd
gd_columns = df_gd.columns.tolist()

# 8. Thêm các cột còn thiếu và gán giá trị rỗng
for col in gd_columns:
    if col not in df_checkpoint_copy.columns:
        df_checkpoint_copy[col] = ""

# 9. Gán giá trị mặc định
df_checkpoint_copy['post_status'] = 'publish'
df_checkpoint_copy['post_category'] = ',249,'
df_checkpoint_copy['default_category'] = '249'
df_checkpoint_copy['featured'] = '0'

# 10. Sắp xếp lại đúng thứ tự cột
df_checkpoint_copy = df_checkpoint_copy[gd_columns]

# 12. Lưu file đã chuẩn hoá
df_checkpoint_copy.to_csv("ger_standardized.csv", index=False)

# 13. (Tuỳ chọn) Xem thử kết quả
print(df_checkpoint_copy[['street', 'zip', 'city', 'country']].head())

                            street    zip      city  country
0         Lausitzer Str. 20, 02828  02828   Görlitz  Germany
1  Hildegard-Burjan-Platz 1, 02826  02826   Görlitz  Germany
2          Berliner Str. 17, 02826  02826   Görlitz  Germany
3      Brautwiesenstraße 04, 02826  02826   Görlitz  Germany
4           Bahnhofstraße 2, 02791  02791  Oderwitz  Germany


In [16]:
df_checkpoint_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7525 entries, 0 to 7524
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                7525 non-null   object 
 1   post_title        7525 non-null   object 
 2   post_content      7525 non-null   object 
 3   post_status       7525 non-null   object 
 4   post_author       7525 non-null   object 
 5   post_type         7525 non-null   object 
 6   post_date         7525 non-null   object 
 7   post_modified     7525 non-null   object 
 8   post_tags         7525 non-null   object 
 9   post_category     7525 non-null   object 
 10  default_category  7525 non-null   object 
 11  featured          7525 non-null   object 
 12  street            7525 non-null   object 
 13  street2           7525 non-null   object 
 14  city              7525 non-null   object 
 15  region            7525 non-null   object 
 16  country           7525 non-null   object 


In [17]:
# in ra 5 giá trị đầu cột lat và long
print(df_checkpoint_copy[['latitude', 'longitude']].head())

    latitude  longitude
0  51.170768  14.978383
1  51.156261  14.982172
2  51.149755  14.982371
3  51.151013  14.969034
4  50.952569  14.726453


In [18]:
duplicates = df_checkpoint_copy[df_checkpoint_copy.duplicated(keep=False)]

In [19]:
duplicate_count = duplicates.shape[0]
print(f"Số lượng bản ghi trùng lặp: {duplicate_count}")    


Số lượng bản ghi trùng lặp: 956


In [20]:
# Đếm số lượng giá trị không null cho từng dòng
df_checkpoint_copy['non_null_count'] = df_checkpoint_copy.notnull().sum(axis=1)

# Sắp xếp theo các cột và theo số lượng giá trị không null giảm dần
df_sorted = df_checkpoint_copy.sort_values(by=['post_title', 'latitude', 'longitude', 'street', 'non_null_count'], ascending=[True, True, True, True, False])

# Xóa các dòng trùng hoàn toàn, giữ lại dòng có nhiều thông tin nhất
df_deduplicated = df_sorted.drop_duplicates(keep='first').drop(columns=['non_null_count'])

# Lưu kết quả ra file mới
output_path = "ger_no_dup.csv"
df_deduplicated.to_csv(output_path, index=False)


In [21]:
df_deduplicated.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6793 entries, 5402 to 3565
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                6793 non-null   object 
 1   post_title        6793 non-null   object 
 2   post_content      6793 non-null   object 
 3   post_status       6793 non-null   object 
 4   post_author       6793 non-null   object 
 5   post_type         6793 non-null   object 
 6   post_date         6793 non-null   object 
 7   post_modified     6793 non-null   object 
 8   post_tags         6793 non-null   object 
 9   post_category     6793 non-null   object 
 10  default_category  6793 non-null   object 
 11  featured          6793 non-null   object 
 12  street            6793 non-null   object 
 13  street2           6793 non-null   object 
 14  city              6793 non-null   object 
 15  region            6793 non-null   object 
 16  country           6793 non-null   object 
 1

Top 200 từ khóa phổ biến và tần suất

In [25]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
# Kết hợp nội dung từ 2 cột post_title và post_content
text_data = df_deduplicated[['post_title']].fillna('').agg(' '.join, axis=1)

# Khởi tạo CountVectorizer để trích xuất từ khóa
vectorizer = CountVectorizer(stop_words='english', max_features=200)
X = vectorizer.fit_transform(text_data)

# Lấy ra từ và tần suất
keywords = vectorizer.get_feature_names_out()
frequencies = np.asarray(X.sum(axis=0)).flatten()

# Tạo dataframe kết quả
keywords_df = pd.DataFrame({'keyword': keywords, 'frequency': frequencies}).sort_values(by='frequency', ascending=False)

In [23]:
# Lưu kết quả vào file CSV
output_keywords_path = "ger_keywords.csv"
keywords_df.to_csv(output_keywords_path, index=False)

In [26]:
# Kiểm tra độ chính xác 
positive_keywords = [
    'vietnam', 'viet', 'việt', 'pho', 'bún', 'nem', 'saigon', "phở", 'sài gòn', 'hà nội', 'hanoi', 'bánh mì',
    'halong', 'huế', 'bánh', 'goi cuon', 'bun cha', 'banh', 'thang', 'nam', 'sen', 'hoan kiem', 'wietnam', 'vietnamese',
    'sapa', 'tre', 'ha long', 'ha noi', 'sai gon', 'sajgon', 'hoang', 'ha-noi', 'com tam', 'hai', 'hoan', 'bami',
    'long', 'binh', 'banh mi', 'sao mai', 'song lam', 'ngoc', 'phuong dong', 'linh', 'vietnamská', 'vietnameské', 'quán', 'anh',
    'vietnamskou', 'vietnamské jídlo', 'vietnamská restaurace', 'ngon', 'hoi an', 'quan', 'vina', 'bếp', 'long', 'nón', 'hà', 
    'vietfood', 'gao', 'mì', 'mộc', 'mai', 'thanh', 'cà', 'tuan', 'lá', 'rong', 'vietnamskou', 'vietnamu', 'vietnamesisches', 'chả',
    'vietnamesische küche', 'vietnamesisch', 'vietnamesischen'
]
negative_keywords = [
    'chinese', 'thai', 'japan', 'korean', 'fusion', 'asia', 'china','india', 'ramen', 'pasta', 'pizza', 'burger',
    'sushi', 'tapas', 'mexican', 'indian', 'kebab', 'italian', 'curry', "tai wan", "singapore", "malaysia", "korea",
    "hong kong", 'resort', 'hotel', 'pub', 'cafe', 'coffee', 'steak', 'park', 'inn', 'post', 'market', 'hall', 'bbq',
    'library', 'sandwich', 'cantonese', 'peking', 'thajská', 'thajské', 'banyan', 'guty', 'shanghai', 'shi', 'pizzerie',
    'bubble', 'kyoto', "mongolian", "indochine", 'nail', 'spa', 'massage', 'laden', 'shop', 'store', 'beauty', 'thailändische',
    'asiatisch', 'asiatischen'
]

# Bước 3: Tạo regex pattern
pattern_positive = re.compile('|'.join(positive_keywords), re.IGNORECASE)
pattern_negative = re.compile('|'.join(negative_keywords), re.IGNORECASE)

# Bước 4: Hàm gán tag
def tag_positive(text):
    if pd.isna(text):
        return ''
    return 'Vietnamese restaurant' if pattern_positive.search(text) else ''

def tag_negative(text):
    if pd.isna(text):
        return ''
    return 'Others' if pattern_negative.search(text) else ''

# Bước 5: Gán PositiveTag và NegativeTag
df_deduplicated['Pos'] = df_deduplicated.apply(
    lambda row: tag_positive(row['post_title']) or tag_positive(row['post_content']),
    axis=1
)

df_deduplicated['Neg'] = df_deduplicated.apply(
    lambda row: tag_negative(row['post_title']) or tag_negative(row['post_content']),
    axis=1
)

# Bước 6: Logic gán ReCheck?
def final_recheck_tag(row):
    if row['Pos'] != '' and row['Neg'] == '':
        return 'N'
    elif row['Pos'] == '' and row['Neg'] != '':
        return 'N'
    elif row['Pos'] == '' and row['Neg'] == '':
        return 'Y'
    else:
        return 'Y'

df_deduplicated['ReCheck?'] = df_deduplicated.apply(final_recheck_tag, axis=1)

In [27]:
# Tạo label mẫu
# Tạo 1 cột mới tên là rỗng mới là Y trong df_checkpoint
def assign_label(row):
    pos = row['Pos'] == 'Vietnamese restaurant'
    neg = row['Neg'] == 'Others'
    recheck = row['ReCheck?']

    if pos and not neg and recheck == 'N':
        return 1
    elif neg and not pos and recheck == 'N':
        return 0
    elif pos and neg and recheck == 'Y':
        return 0
    else:
        return ''

df_deduplicated['Y'] = df_deduplicated.apply(assign_label, axis=1)

df = df_deduplicated.copy()

# Xuất file để check manual
columns_to_export = [
    'post_title', 'post_content', 'website', 'google_profile', "Y", 'city'
]
df = df[columns_to_export]
df.to_csv('ger_labeled.csv', index=False)


In [28]:
df_deduplicated.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6793 entries, 5402 to 3565
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                6793 non-null   object 
 1   post_title        6793 non-null   object 
 2   post_content      6793 non-null   object 
 3   post_status       6793 non-null   object 
 4   post_author       6793 non-null   object 
 5   post_type         6793 non-null   object 
 6   post_date         6793 non-null   object 
 7   post_modified     6793 non-null   object 
 8   post_tags         6793 non-null   object 
 9   post_category     6793 non-null   object 
 10  default_category  6793 non-null   object 
 11  featured          6793 non-null   object 
 12  street            6793 non-null   object 
 13  street2           6793 non-null   object 
 14  city              6793 non-null   object 
 15  region            6793 non-null   object 
 16  country           6793 non-null   object 
 1

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6793 entries, 5402 to 3565
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   post_title      6793 non-null   object
 1   post_content    6793 non-null   object
 2   website         4632 non-null   object
 3   google_profile  6793 non-null   object
 4   Y               6793 non-null   object
 5   city            6793 non-null   object
dtypes: object(6)
memory usage: 371.5+ KB


In [30]:
# Đọc file labeled
df_labeled = pd.read_csv('ger_labeled.csv')
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_labeled['Y'].value_counts(dropna=False))

Y
0.0    4386
1.0    1365
NaN    1042
Name: count, dtype: int64


In [31]:
print(df_deduplicated['Y'].value_counts(dropna=False))

Y
0    4386
1    1365
     1042
Name: count, dtype: int64


In [32]:
# Kiểm tra lại nhanh số lượng giá trị thiếu (NaN) sau khi chuyển đổi
missing_summary = df_deduplicated.isna().sum()
missing_summary

ID                     0
post_title             0
post_content           0
post_status            0
post_author            0
post_type              0
post_date              0
post_modified          0
post_tags              0
post_category          0
default_category       0
featured               0
street                 0
street2                0
city                   0
region                 0
country                0
zip                    0
latitude               0
longitude              0
website             2161
neighbourhood          0
facebook            6453
instagram           6723
twitter             6793
phone                178
email                  0
logo                   0
google_profile         0
post_images            0
Pos                    0
Neg                    0
ReCheck?               0
Y                      0
dtype: int64

In [33]:
# Chuyển toàn bộ chuỗi rỗng hoặc chuỗi chỉ chứa khoảng trắng thành NaN
df_final = df_deduplicated.copy()

In [34]:
df_final.head(5)

,ID,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
5402,,"""Asia Bistro"" Vietnam - Thailand - China","""Asia Bistro"" Vietnam - Thailand - China locat...",publish,,,,,,",249,",...,NaN,+493459495658,,,https://maps.google.com/?q=place_id:ChIJw2TOqq...,,Vietnamese restaurant,Others,Y,0
35,,"""Cay Tre"" Bambus Restaurant.","""Cay Tre"" Bambus Restaurant. located in Tesche...",publish,,,,,,",249,",...,NaN,+4935714509866,,,https://maps.google.com/?q=place_id:ChIJPRGPv7...,,Vietnamese restaurant,,N,1
381,,"""LION KING"" + Sushi in Murnau",Di. - Sa. :11:00 - 14:30 & 17:00 - 22:00 | Mon...,publish,,,,,,",249,",...,NaN,+4988416286327,,,https://maps.google.com/?q=place_id:ChIJn-DJ-j...,,Vietnamese restaurant,Others,Y,0
5206,,"""My Sen"" Vietnamesisches Restaurant + Biergarten",Auf dieser Website finden Sie die Speisekarte ...,publish,,,,,,",249,",...,NaN,+493451206932,,,https://maps.google.com/?q=place_id:ChIJ9Tv19D...,,Vietnamese restaurant,,N,1
7,,"""Phố Việt"" Restaurant & Bar",Phố Việt Restaurant,publish,,,,,,",249,",...,NaN,+4934199391747,,,https://maps.google.com/?q=place_id:ChIJFTmbdi...,,Vietnamese restaurant,,N,1


In [35]:
# Gán giá trị mặc định nếu thiếu
df_final['post_type'] = df_final['post_type'].fillna('gd_place')
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
default_date = '2025-06-01 00:00:00'
df_final['post_date'] = df_final['post_date'].fillna(default_date)
df_final['post_modified'] = df_final['post_modified'].fillna(default_date)
df_final['post_author'] = df_final['post_author'].fillna('admin')
df_final["post_content"] = ""
df_final = df_final.replace(r'^\s*$', pd.NA, regex=True)
df_final.set_index('ID', inplace=True)
# Đổi tên cột id thành ID
df_final.head(5)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6793 entries, <NA> to <NA>
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        6793 non-null   object 
 1   post_content      0 non-null      object 
 2   post_status       6793 non-null   object 
 3   post_author       0 non-null      object 
 4   post_type         0 non-null      object 
 5   post_date         0 non-null      object 
 6   post_modified     0 non-null      object 
 7   post_tags         0 non-null      object 
 8   post_category     6793 non-null   object 
 9   default_category  6793 non-null   object 
 10  featured          6793 non-null   object 
 11  street            6793 non-null   object 
 12  street2           0 non-null      object 
 13  city              6792 non-null   object 
 14  region            0 non-null      object 
 15  country           6793 non-null   object 
 16  zip               6792 non-null   object 
 1

In [36]:
df_final.head(5)
df_final.to_csv('ger_final.csv', index=False)

In [38]:
%pip install openpyxl --quiet

# Xuất file excel
output_excel_path = 'ger_final.xlsx'
df_final.to_excel(output_excel_path, index=False)

Note: you may need to restart the kernel to use updated packages.


In [39]:
df_final.head(10)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
ID,,,,,,,,,,,,,,,,,,,,,
<NA>,"""Asia Bistro"" Vietnam - Thailand - China",<NA>,publish,<NA>,<NA>,<NA>,<NA>,<NA>,",249,",249,...,NaN,+493459495658,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJw2TOqq...,<NA>,Vietnamese restaurant,Others,Y,0
<NA>,"""Cay Tre"" Bambus Restaurant.",<NA>,publish,<NA>,<NA>,<NA>,<NA>,<NA>,",249,",249,...,NaN,+4935714509866,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJPRGPv7...,<NA>,Vietnamese restaurant,<NA>,N,1
<NA>,"""LION KING"" + Sushi in Murnau",<NA>,publish,<NA>,<NA>,<NA>,<NA>,<NA>,",249,",249,...,NaN,+4988416286327,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJn-DJ-j...,<NA>,Vietnamese restaurant,Others,Y,0
<NA>,"""My Sen"" Vietnamesisches Restaurant + Biergarten",<NA>,publish,<NA>,<NA>,<NA>,<NA>,<NA>,",249,",249,...,NaN,+493451206932,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJ9Tv19D...,<NA>,Vietnamese restaurant,<NA>,N,1
<NA>,"""Phố Việt"" Restaurant & Bar",<NA>,publish,<NA>,<NA>,<NA>,<NA>,<NA>,",249,",249,...,NaN,+4934199391747,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJFTmbdi...,<NA>,Vietnamese restaurant,<NA>,N,1
<NA>,"""Tinh Asia"" - Tinh Nagel",<NA>,publish,<NA>,<NA>,<NA>,<NA>,<NA>,",249,",249,...,NaN,+493654212183,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJO25YfP...,<NA>,<NA>,Others,N,0
<NA>,"""Zum Weinkeller""",<NA>,publish,<NA>,<NA>,<NA>,<NA>,<NA>,",249,",249,...,NaN,+4967211867994,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJ6zVWWm...,<NA>,Vietnamese restaurant,<NA>,N,1
<NA>,- Eiscafé Riva,<NA>,publish,<NA>,<NA>,<NA>,<NA>,<NA>,",249,",249,...,NaN,+493413376580,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJz-dwOa...,<NA>,<NA>,<NA>,Y,<NA>
<NA>,030 Ronin - Fusion & Vegan Sushi,<NA>,publish,<NA>,<NA>,<NA>,<NA>,<NA>,",249,",249,...,NaN,+4982129716584,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJ3RaRp1...,<NA>,<NA>,Others,N,0


In [42]:
# đọc file labeled_new
#df_labeled_new = pd.read_csv('ger_labeled_new.csv')
df_standardized= pd.read_csv('ger_final.csv')
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_standardized['Y'].value_counts(dropna=False))

Y
0.0    4386
1.0    1365
NaN    1042
Name: count, dtype: int64


In [ ]:
# Bổ sung giá trị cột Y từ df_labeled_new vào df_standardized dựa trên index (không có cột ID)
df_standardized['Y'] = df_labeled_new['Y'].values
# Lưu kết quả vào file CSV
output_path = 'ger_standardized_with_labels.csv'
# đọc poland_standardized_with_labels.csv
df_standardized.to_csv(output_path, index=False)
df_standardized_with_labels = pd.read_csv(output_path)  
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_standardized_with_labels['Y'].value_counts(dropna=False))
# Lưu lại file đã chuẩn hoá
df_standardized_with_labels.to_csv('ger_standardized_final.csv', index=False)
# in head 5 dòng
df_standardized_with_labels.head(5)

Y
1.0    526
0.0    375
NaN    301
Name: count, dtype: int64


,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
0,123 Viet Thai Restaurant,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,+420606857588,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ605V-F...,NaN,Vietnamese restaurant,Others,Y,0.0
1,1989Food - Grill & Sushi Bar Sluknov,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,+420776331389,NaN,NaN,https://maps.google.com/?q=place_id:ChIJIdxGjP...,NaN,NaN,Others,N,0.0
2,1994 Asian Restaurant - Autentická chuť asijsk...,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,+420776823203,NaN,NaN,https://maps.google.com/?q=place_id:ChIJqwRml3...,NaN,NaN,Others,N,0.0
3,2Kyo Asian Restaurant,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,+420778558848,NaN,NaN,https://maps.google.com/?q=place_id:ChIJg37Lbo...,NaN,Vietnamese restaurant,Others,Y,0.0
4,7days,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,+420774089399,NaN,NaN,https://maps.google.com/?q=place_id:ChIJO6J2cW...,NaN,NaN,NaN,Y,NaN


In [43]:
# Gán giá trị mặc định nếu thiếu
df_standardized['post_type'] = df_standardized['post_type'].fillna('gd_place')
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
default_date = '2025-05-31 00:00:00'
df_standardized['post_date'] = df_standardized['post_date'].fillna(default_date)
df_standardized['post_modified'] = df_standardized['post_modified'].fillna(default_date)
df_standardized['post_author'] = df_standardized['post_author'].fillna('admin')
df_standardized["post_content"] = ""
df_standardized = df_standardized.replace(r'^\s*$', pd.NA, regex=True)
# trích xuất file cuối chỉ có các dòng mà giá trị cột Y là 1 và xóa cột Y sau đó 
df_standardized = df_standardized[df_standardized['Y'] == 1]
df_standardized.drop(columns=['Y'], inplace=True)
df_standardized.head(5)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,instagram,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?
1,"""Cay Tre"" Bambus Restaurant.",<NA>,publish,admin,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,NaN,",249,",249,...,NaN,NaN,+4935714509866,NaN,NaN,https://maps.google.com/?q=place_id:ChIJPRGPv7...,NaN,Vietnamese restaurant,NaN,N
3,"""My Sen"" Vietnamesisches Restaurant + Biergarten",<NA>,publish,admin,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,NaN,",249,",249,...,NaN,NaN,+493451206932,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ9Tv19D...,NaN,Vietnamese restaurant,NaN,N
4,"""Phố Việt"" Restaurant & Bar",<NA>,publish,admin,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,NaN,",249,",249,...,NaN,NaN,+4934199391747,NaN,NaN,https://maps.google.com/?q=place_id:ChIJFTmbdi...,NaN,Vietnamese restaurant,NaN,N
6,"""Zum Weinkeller""",<NA>,publish,admin,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,NaN,",249,",249,...,NaN,NaN,+4967211867994,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ6zVWWm...,NaN,Vietnamese restaurant,NaN,N
10,12S Pho Weimar - Ecke Vietnam,<NA>,publish,admin,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,NaN,",249,",249,...,NaN,NaN,+4936432529141,NaN,NaN,https://maps.google.com/?q=place_id:ChIJncFhqA...,NaN,Vietnamese restaurant,NaN,N


In [46]:
# So sánh định dạng từng cột của df_true với df_gd
def compare_column_formats(df1, df2):
    comparison = {}
    for col in df1.columns:
        if col in df2.columns:
            comparison[col] = {
                'df1_dtype': df1[col].dtype,
                'df2_dtype': df2[col].dtype,
                'df1_unique_count': df1[col].nunique(),
                'df2_unique_count': df2[col].nunique()
            }
        else:
            comparison[col] = {
                'df1_dtype': df1[col].dtype,
                'df2_dtype': None,
                'df1_unique_count': df1[col].nunique(),
                'df2_unique_count': None
            }
    return comparison
# So sánh định dạng cột của df_true với df_gd
comparison_result = compare_column_formats(df_standardized, df_gd)
# In kết quả so sánh
for col, info in comparison_result.items():
    print(f"Cột: {col}")
    print(f"  - df_true dtype: {info['df1_dtype']}, unique count: {info['df1_unique_count']}")
    print(f"  - df_gd dtype: {info['df2_dtype']}, unique count: {info['df2_unique_count']}")
    print()
# Ép kiểu các cột trong df_true để phù hợp với df_gd
def convert_column_types(df, reference_df):
    for col in reference_df.columns:
        if col in df.columns:
            ref_dtype = reference_df[col].dtype
            if ref_dtype == 'object':
                df[col] = df[col].astype(str)
            elif ref_dtype == 'int64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            elif ref_dtype == 'float64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0).astype(float)
            elif ref_dtype == 'datetime64[ns]':
                df[col] = pd.to_datetime(df[col], errors='coerce')
    return df   
# Chuyển đổi kiểu dữ liệu của df_true để phù hợp với df_gd
df_standardized = convert_column_types(df_standardized, df_gd)

Cột: post_title
  - df_true dtype: object, unique count: 1250
  - df_gd dtype: object, unique count: 100

Cột: post_content
  - df_true dtype: object, unique count: 0
  - df_gd dtype: object, unique count: 19

Cột: post_status
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: post_author
  - df_true dtype: object, unique count: 1
  - df_gd dtype: int64, unique count: 19

Cột: post_type
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: post_date
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 100

Cột: post_modified
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 100

Cột: post_tags
  - df_true dtype: float64, unique count: 0
  - df_gd dtype: object, unique count: 1

Cột: post_category
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: default_category
  - df_true dtype: int64, unique count: 1
  - df_gd

In [47]:
df_standardized.head(5)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,instagram,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?
1,"""Cay Tre"" Bambus Restaurant.",<NA>,publish,0,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,nan,",249,",249,...,nan,nan,+4935714509866,nan,nan,https://maps.google.com/?q=place_id:ChIJPRGPv7...,0.0,Vietnamese restaurant,NaN,N
3,"""My Sen"" Vietnamesisches Restaurant + Biergarten",<NA>,publish,0,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,nan,",249,",249,...,nan,nan,+493451206932,nan,nan,https://maps.google.com/?q=place_id:ChIJ9Tv19D...,0.0,Vietnamese restaurant,NaN,N
4,"""Phố Việt"" Restaurant & Bar",<NA>,publish,0,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,nan,",249,",249,...,nan,nan,+4934199391747,nan,nan,https://maps.google.com/?q=place_id:ChIJFTmbdi...,0.0,Vietnamese restaurant,NaN,N
6,"""Zum Weinkeller""",<NA>,publish,0,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,nan,",249,",249,...,nan,nan,+4967211867994,nan,nan,https://maps.google.com/?q=place_id:ChIJ6zVWWm...,0.0,Vietnamese restaurant,NaN,N
10,12S Pho Weimar - Ecke Vietnam,<NA>,publish,0,gd_place,2025-05-31 00:00:00,2025-05-31 00:00:00,nan,",249,",249,...,nan,nan,+4936432529141,nan,nan,https://maps.google.com/?q=place_id:ChIJncFhqA...,0.0,Vietnamese restaurant,NaN,N


In [48]:
# Kiểm tra trùng post_title giữa df_standardized_with_labels và df_gd
def check_duplicates(df1, df2, column):
    duplicates = df1[df1[column].isin(df2[column])]
    return duplicates
# Kiểm tra trùng post_title
duplicates = check_duplicates(df_standardized, df_gd, 'post_title')
# In ra số lượng bản ghi trùng lặp
print(f"Số lượng bản ghi trùng lặp trong cột 'post_title': {duplicates.shape[0]}")
# In ra 5 bản ghi trùng lặp
print(duplicates[['post_title']].head(5))

Số lượng bản ghi trùng lặp trong cột 'post_title': 2
      post_title
4636  Pho Ha Noi
4637  Pho Ha Noi


In [49]:
# Xóa các bản ghi trùng lặp trong df_standardized_with_labels
df_standardized = df_standardized[~df_standardized['post_title'].isin(duplicates['post_title'])]
# In ra số lượng bản ghi sau khi xóa trùng lặp
print(f"Số lượng bản ghi sau khi xóa trùng lặp: {df_standardized.shape[0]}")

Số lượng bản ghi sau khi xóa trùng lặp: 1363


In [51]:
df_standardized.head(5)
#Xóa cột neg, pos, recheck
df_standardized.drop(columns=['Neg', 'Pos', 'ReCheck?'], inplace=True)
# In ra thông tin của các cột
df_standardized.info()
# Lưu lại file đã chuẩn hoá
df_standardized.to_csv('ger_upload.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 1363 entries, 1 to 6788
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        1363 non-null   object 
 1   post_content      1363 non-null   object 
 2   post_status       1363 non-null   object 
 3   post_author       1363 non-null   int64  
 4   post_type         1363 non-null   object 
 5   post_date         1363 non-null   object 
 6   post_modified     1363 non-null   object 
 7   post_tags         1363 non-null   object 
 8   post_category     1363 non-null   object 
 9   default_category  1363 non-null   int64  
 10  featured          1363 non-null   int64  
 11  street            1363 non-null   object 
 12  street2           1363 non-null   float64
 13  city              1363 non-null   object 
 14  region            1363 non-null   object 
 15  country           1363 non-null   object 
 16  zip               1363 non-null   object 
 17  

In [61]:
import pandas as pd

# 1. Load dữ liệu
df_sample = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv")

# 2. Thêm cột ID từ index (bắt đầu từ 1)
df_slovakia['ID'] = df_slovakia.index + 1
df_slovakia = df_slovakia[['ID'] + [col for col in df_slovakia.columns if col != 'ID']]

# 3. Đảm bảo các cột đúng thứ tự như file mẫu
df_slovakia = df_slovakia[df_sample.columns]

df_slovakia["post_type"] = "gd_place"
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
df_slovakia['post_date'] = '2025-05-31 00:00:00'
df_slovakia['post_modified'] = '2025-05-31 00:00:00'
df_slovakia['post_author'] = 'admin'

# 4. Chuyển kiểu dữ liệu theo sample
for col in df_sample.columns:
    ref_dtype = df_sample[col].dtype
    if ref_dtype == 'object':
        df_slovakia[col] = df_slovakia[col].astype(str)
    elif 'int' in str(ref_dtype):
        df_slovakia[col] = pd.to_numeric(df_slovakia[col], errors='coerce').fillna(0).astype(int)
    elif 'float' in str(ref_dtype):
        df_slovakia[col] = pd.to_numeric(df_slovakia[col], errors='coerce')
    elif 'datetime' in str(ref_dtype):
        df_slovakia[col] = pd.to_datetime(df_slovakia[col], errors='coerce')

# 5. Làm sạch các chuỗi rỗng hoặc chứa 'nan', 'none'
df_slovakia = df_slovakia.replace(r'^\s*$', pd.NA, regex=True)
df_slovakia = df_slovakia.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)

df_slovakia['region'] = df_slovakia['region'].apply(
    lambda x: "-" if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none', 'n/a'] else x
)
# Chuẩn hóa zip code
df_slovakia['zip'] = df_slovakia['zip'].astype(str).str.replace(r'\.0$', '', regex=True)
df_slovakia['zip'] = df_slovakia['zip'].apply(lambda x: x.zfill(5) if x.isdigit() else x)
# Làm sạch street2 để không có 0.0 hoặc NaN
df_slovakia['street2'] = df_slovakia['street2'].apply(
    lambda x: pd.NA if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none'] else x
)

# 6. Chuẩn hóa số điện thoại
df_slovakia['phone'] = df_slovakia['phone'].astype(str).str.replace(r'\.0$', '', regex=True)
df_slovakia['phone'] = df_slovakia['phone'].apply(lambda x: '+' + x if isinstance(x, str) and x and not x.startswith('+') else x)

df_slovakia.drop(columns=['ID'], inplace=True)
# 8. Xuất ra file CSV
df_slovakia.to_csv("poland_final_upload_ready.csv", index=False)


C:\Users\Nhung\AppData\Local\Temp\ipykernel_2552\2675451420.py:33: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_slovakia = df_slovakia.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)


In [62]:
df_slovakia.info()

<class 'pandas.core.frame.DataFrame'>
Index: 650 entries, 0 to 651
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        650 non-null    object 
 1   post_content      0 non-null      object 
 2   post_status       650 non-null    object 
 3   post_author       650 non-null    int64  
 4   post_type         650 non-null    object 
 5   post_date         650 non-null    object 
 6   post_modified     650 non-null    object 
 7   post_tags         0 non-null      object 
 8   post_category     650 non-null    object 
 9   default_category  650 non-null    int64  
 10  featured          650 non-null    int64  
 11  street            650 non-null    object 
 12  street2           0 non-null      object 
 13  city              645 non-null    object 
 14  region            650 non-null    object 
 15  country           650 non-null    object 
 16  zip               650 non-null    object 
 17  la

In [59]:

# 1. Load dữ liệu
df_slovakia = pd.read_csv(r'C:\Users\Nhung\Downloads\We_Love_Pho\poland\poland_final.csv')  # hoặc đường dẫn thực tế
df_sample = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv")
# Kiểm tra trùng post_title giữa df_standardized_with_labels và df_gd
def check_duplicates(df1, df2, column):
    duplicates = df1[df1[column].isin(df2[column])]
    return duplicates
# Kiểm tra trùng post_title
duplicates = check_duplicates(df_slovakia, df_sample, 'post_title')
# In ra số lượng bản ghi trùng lặp
print(f"Số lượng bản ghi trùng lặp trong cột 'post_title': {duplicates.shape[0]}")
# In ra 5 bản ghi trùng lặp
print(duplicates[['post_title']].head(5))

Số lượng bản ghi trùng lặp trong cột 'post_title': 2
    post_title
21    Mama Pho
140    Pho Mai


In [60]:
# Xóa các bản ghi trùng lặp trong df_standardized_with_labels
df_slovakia = df_slovakia[~df_slovakia['post_title'].isin(duplicates['post_title'])]
# In ra số lượng bản ghi sau khi xóa trùng lặp
print(f"Số lượng bản ghi sau khi xóa trùng lặp: {df_slovakia.shape[0]}")

Số lượng bản ghi sau khi xóa trùng lặp: 650
